# 15 — Capstone: research → synthesis → review

This is the default production shape for a substantial knowledge task: validate, route, fan out to independent workers, synthesize, verify, repair, checkpoint, and pause before consequential publication. Every source is synthetic and the run is deterministic, so every transition can be inspected.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'notebooks'))
from lab_core.patterns import ApprovalGate, Budget, CheckpointStore, evaluate_optimize, fan_out_fan_in, route

request = 'Compare two agentic architecture options and recommend one.'
budget = Budget(max_steps=8, max_retries=2, max_spend=0.50)
store = CheckpointStore()

def classify(text):
    return 'architecture' if 'architecture' in text.lower() else 'general'

def worker(source):
    budget.charge(spend=0.02)
    return {'source': source, 'finding': f'{source} favors explicit verification and bounded autonomy'}

route_info = route(request, classify, {'architecture': lambda _: 'architecture-review', 'general': lambda _: 'general-review'})
evidence = fan_out_fan_in(['official docs', 'tests', 'design notes'], worker, lambda rows: rows, workers=3)
draft = 'Recommendation: use a deterministic workflow spine. ' + ' '.join(row['finding'] for row in evidence)

def evaluate(candidate):
    return {'pass': 'Recommendation:' in candidate and len(evidence) == 3, 'issues': []}

def revise(candidate, report):
    budget.charge(spend=0.01, retry=True)
    return candidate + ' Evidence coverage verified.'

final, reports = evaluate_optimize(draft, evaluate, revise, max_rounds=2)
store.save('capstone-1', {'route': route_info, 'evidence': evidence, 'final': final, 'reports': reports})
approved = ApprovalGate(lambda proposal: proposal['risk'] == 'low').require({'artifact': final, 'risk': 'low'})
print(approved['artifact'])
print('checkpoint keys:', store.load('capstone-1').keys())
print('budget:', budget)